In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, input_file_name
from src.utils.config_loader import ConfigLoader

def ingest_pos_transactions(spark: SparkSession, config: ConfigLoader):
    raw_path = f"{config.get('storage.base_path')}{config.get('sources.pos_csv.transactions_path')}"
    checkpoint_path = f"{config.get('storage.checkpoint_base')}/raw_transactions"
    target_table = f"{config.get('catalog')}.{config.get('targets.schemas.bronze')}.pos_transactions"

    df = (spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "csv")
          .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(raw_path)
          .withColumn("_ingested_at", current_timestamp())
          .withColumn("_source_file", input_file_name()))

    query = (df.writeStream
             .format("delta")
             .outputMode("append")
             .option("checkpointLocation", checkpoint_path)
             .toTable(target_table))
    
    return query